In [0]:
dbutils.widgets.removeAll()

In [0]:
# DBTITLE 2, Environment and Dependency Setup
import os
import sys
from pathlib import Path

# Force workspace file system synchronization
os.sync()

# Set absolute workspace directory boundaries
ROOT_DIR = Path("/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing")
sys.path.append(str(ROOT_DIR))

In [0]:
# DBTITLE 1, Define Global Pipeline Parameters
dbutils.widgets.text(
    "GoldConfigPath", 
    "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/DimQualityEvent/Gold/Config/dimQualityEvent.json", 
    "Target Gold Config JSON Path"
)

config_path = dbutils.widgets.get("GoldConfigPath")

In [0]:
# DBTITLE 3, Gold Layer Execution Function
def trigger_gold_processing(config_file_path: str):
    """Triggers the generalized Gold engine using the targeted JSON configuration metadata matrix."""
    gold_notebook_path = f"{ROOT_DIR}/DimQualityEvent/Gold/Notebooks/GenericSubGroupProcessing"
    
    print(f"--> Invoking Gold Layer Engine processing metadata matrix...")
    print(f"    Target Configuration File: {config_file_path}")
    
    payload_arguments = {
        "SubGroupConfigPath": config_file_path
    }
    
    # Delegate execution handling to the Generic processing notebook
    result = dbutils.notebook.run(gold_notebook_path, 1200, arguments=payload_arguments)
    print(f"    Gold Layer Response: {result}")
    return result

In [0]:
# DBTITLE 4, Main Pipeline Sequenced Loop Execution
def main():
    print(f"=======================================================")
    print(f"STARTING LIFECYCLE FOR PIPELINE: QualityEvent_Gold")
    print(f"=======================================================")
    
    try:
        # Step 1: Run downstream Gold parsing table conversions and merge modifications
        pipeline_result = trigger_gold_processing(config_path)
        
        print("\n=======================================================")
        print("PIPELINE PROCESS COMPLETE")
        print(f"Final Outcome: {pipeline_result}")
        print("=======================================================")
        
        dbutils.notebook.exit(f"SUCCESS: {pipeline_result}")
        
    except Exception as e:
        error_msg = f"Pipeline execution faulted during processing: {str(e)}"
        print(f"CRITICAL: {error_msg}")
        dbutils.notebook.exit(f"FAILED: {error_msg}")

In [0]:
if __name__ == "__main__":
    main()